# 활력징후 → 조기경보 파이프라인 워크스루 (PhysioNet Challenge 2019)

이 노트북은 **데이터 전처리 → EDA 시각화 → XGBoost(vs NEWS) → LSTM/GRU 학습(Loss 곡선)** 을
단계별로 눈으로 확인하기 위한 것입니다. PowerShell 텍스트 출력 대신 그래프로 진행 상황을 봅니다.

> ⚠️ **데이터 성격**: Challenge 2019는 **패혈증(sepsis)** 조기예측 데이터입니다. 우리 본 주제인
> **심정지(cardiac arrest)** 와는 다르지만, "활력징후 시계열 → 임박한 악화 이벤트"라는 **문제 구조가
> 같아** 파이프라인을 실데이터에서 검증하는 **프록시**로 씁니다. 여기 숫자는 심정지 성능이 아니라
> "파이프라인이 실데이터에서 작동한다"는 근거입니다.

**플롯 제목은 영어**로 둡니다 (서버에 한글 폰트가 없으면 matplotlib에서 □로 깨지기 때문). 설명은
이 마크다운 셀들에 한글로 있습니다.

## 0. 설정 — 경로/임포트

`DATA_DIR`를 서버의 실제 데이터 폴더로 맞추세요.

In [ ]:
import sys
from pathlib import Path

# 이 노트북이 repo/notebooks/ 안에 있다고 가정하고 repo/src 를 import 경로에 추가
REPO = Path.cwd()
if REPO.name == "notebooks":
    REPO = REPO.parent
sys.path.insert(0, str(REPO / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

%matplotlib inline
plt.rcParams["figure.dpi"] = 110

# ============== 설정 (서버 환경에 맞게 수정) ==============
DATA_DIR  = "/workspace/training_setA"   # Challenge 2019 PSV 폴더
MAX_FILES = 1000                          # 읽을 환자 파일 수 (속도용; None=전체)
HORIZON   = 6                             # 예측 지평(시간) — 1h면 양성이 너무 희박함
USE_GPU   = False                         # XGBoost를 GPU로 학습하려면 True
# =========================================================

print("repo :", REPO)
print("data :", DATA_DIR)
VITALS = ["pulse", "sbp", "dbp", "temperature", "spo2", "resp_rate"]

## 1. 데이터 로드

`cohort_from_challenge2019` 가 각 `p*.psv`(환자 1명)를 읽어 6종 활력징후로 매핑하고,
`SepsisLabel==1` 이 처음 뜨는 시각을 이벤트 시각(`arrest_hour`)으로 잡습니다. 이벤트가 없는 환자는
대조군(`arrest_hour = NaN`)입니다. 값 정제(`sanitize_vitals`)도 이 단계에서 적용됩니다.

In [ ]:
from vitals_data import cohort_from_challenge2019

cohort = cohort_from_challenge2019(DATA_DIR, max_files=MAX_FILES)

n_patients = cohort.vitals["patient_id"].nunique()
n_event = int(cohort.events["arrest_hour"].notna().sum())
print(f"환자 수: {n_patients}  |  이벤트(sepsis) 환자: {n_event}  |  대조군: {n_patients - n_event}")
print(f"활력징후 행(시점) 수: {len(cohort.vitals):,}")
print("\nvitals 테이블 미리보기:")
cohort.vitals.head()

## 2. EDA 시각화

데이터 '형태'를 눈으로 확인합니다: (a) 활력징후 분포, (b) 결측률, (c) 환자별 기록 길이 + 클래스 균형,
(d) 이벤트 환자 1명의 궤적.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for ax, v in zip(axes.ravel(), VITALS):
    ax.hist(cohort.vitals[v].dropna(), bins=50, color="steelblue")
    ax.set_title(v)
    ax.set_ylabel("count")
fig.suptitle("Vital-sign distributions (after sanitation)", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
miss = cohort.vitals[VITALS].isna().mean().sort_values()
fig, ax = plt.subplots(figsize=(7, 4))
miss.plot.barh(ax=ax, color="indianred")
ax.set_title("Missing-value fraction per vital")
ax.set_xlabel("fraction missing")
for i, val in enumerate(miss.values):
    ax.text(val + 0.005, i, f"{val:.0%}", va="center")
plt.tight_layout()
plt.show()

In [ ]:
lengths = cohort.vitals.groupby("patient_id")["hour"].max() + 1
event_ids = set(cohort.events.loc[cohort.events["arrest_hour"].notna(), "patient_id"])

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].hist(lengths, bins=40, color="slateblue")
ax[0].set_title("Recorded hours per patient")
ax[0].set_xlabel("hours"); ax[0].set_ylabel("patients")

n_ev = len(event_ids); n_ct = n_patients - n_ev
ax[1].bar(["event (sepsis)", "control"], [n_ev, n_ct], color=["crimson", "gray"])
ax[1].set_title("Class balance (patient level)")
for i, val in enumerate([n_ev, n_ct]):
    ax[1].text(i, val, str(val), ha="center", va="bottom")
plt.tight_layout()
plt.show()

In [ ]:
# 이벤트 환자 1명의 활력징후 궤적 (빨간 점선 = 이벤트 발생 시각)
example_id = sorted(event_ids)[0]
g = cohort.vitals[cohort.vitals["patient_id"] == example_id].sort_values("hour")
arrest_h = cohort.events.set_index("patient_id").loc[example_id, "arrest_hour"]

fig, ax = plt.subplots(figsize=(11, 5))
for v in VITALS:
    ax.plot(g["hour"], g[v], marker=".", ms=4, label=v)
ax.axvline(arrest_h, color="red", ls="--", lw=2, label="event onset")
ax.set_title(f"Patient {example_id} — vital-sign trajectory")
ax.set_xlabel("hour"); ax.set_ylabel("value")
ax.legend(ncol=4, fontsize=8)
plt.tight_layout()
plt.show()

## 3. 윈도우 생성 + 예측 지평의 영향

슬라이딩 윈도우를 만들고, 각 윈도우를 "앞으로 H시간 안에 이벤트가 오는가"로 라벨링합니다.
**예측 지평(horizon)이 좁을수록 양성 윈도우가 극단적으로 희박**해집니다 — 지평 1h면 환자당 양성이
사실상 1개뿐이라 양성비율 ~0.2%가 되고, 이게 AUPRC가 base rate로 붕괴하는 원인입니다.
아래 그래프로 그 관계를 직접 봅니다.

In [ ]:
from vitals_data import build_windows

rows = []
for H in [1, 2, 3, 6, 12, 24]:
    w = build_windows(cohort, prediction_horizon_hours=H)
    rows.append((H, len(w.labels), int(w.labels.sum()), float(w.labels.mean())))
tab = pd.DataFrame(rows, columns=["horizon_h", "windows", "positives", "positive_rate"])
print(tab.to_string(index=False))

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(tab["horizon_h"], tab["positive_rate"] * 100, marker="o")
ax.set_xlabel("prediction horizon (h)")
ax.set_ylabel("positive window rate (%)")
ax.set_title("Wider horizon -> more positives (1h collapses AUPRC)")
for _, r in tab.iterrows():
    ax.annotate(f"{r.positive_rate*100:.2f}%", (r.horizon_h, r.positive_rate*100),
                textcoords="offset points", xytext=(0, 6), fontsize=8)
plt.tight_layout()
plt.show()

## 4. XGBoost vs NEWS (지평 = `HORIZON`)

윈도우 통계 + 개인 기저선 이탈 피처를 만들고, **환자 단위 분할**(누수 방지)로 학습합니다.
비교군은 임상 규칙 점수 NEWS. 지표는 AUPRC·ROC·민감도@95%특이도·오경보율·알람수/100.

In [ ]:
from vitals_data import add_personalized_features, patient_level_split
from vitals_train import train_xgboost, evaluate_news_baseline, compute_news_scores

windowed = add_personalized_features(
    build_windows(cohort, prediction_horizon_hours=HORIZON), cohort
)
print(f"[horizon={HORIZON}h] windows={len(windowed.labels):,} "
      f"positives={int(windowed.labels.sum())} "
      f"({windowed.labels.mean()*100:.2f}%)")

split = patient_level_split(windowed)
model, xgb = train_xgboost(split, use_gpu=USE_GPU)
news = evaluate_news_baseline(split)

metrics = pd.DataFrame([{
    "model": m.model_name, "AUPRC": m.auprc, "ROC": m.roc_auc,
    "sens@95spec": m.sensitivity_at_95_specificity,
    "falseAlarm": m.false_alarm_rate, "alarms/100": m.alarms_per_100_windows,
} for m in (xgb, news)])
metrics.round(3)

In [ ]:
from sklearn.metrics import precision_recall_curve, roc_curve

xgb_score = model.predict_proba(split.X_test)[:, 1]
news_score = compute_news_scores(split.X_test)

fig, ax = plt.subplots(1, 2, figsize=(13, 5))
for name, s, c in [("XGBoost", xgb_score, "C0"), ("NEWS", news_score, "C1")]:
    p, r, _ = precision_recall_curve(split.y_test, s)
    ax[0].plot(r, p, label=name, color=c)
    fpr, tpr, _ = roc_curve(split.y_test, s)
    ax[1].plot(fpr, tpr, label=name, color=c)
base = float(np.mean(split.y_test))
ax[0].axhline(base, color="gray", ls=":", label=f"base rate {base:.3f}")
ax[0].set_title("PR curve"); ax[0].set_xlabel("Recall"); ax[0].set_ylabel("Precision"); ax[0].legend()
ax[1].plot([0, 1], [0, 1], "k--", alpha=0.3)
ax[1].set_title("ROC curve"); ax[1].set_xlabel("FPR"); ax[1].set_ylabel("TPR"); ax[1].legend()
plt.tight_layout()
plt.show()

## 5. 딥러닝 벤치마크 (LSTM/GRU) — **Loss 곡선**

같은 데이터를 **환자 단위 시퀀스 분류**로 학습합니다: 입력 = 환자의 활력징후 시계열 `[T, F]`,
라벨 = 그 환자가 이벤트(sepsis)를 겪었는가(1/0). 가변 길이 시퀀스를 padding+masking으로 처리하고,
`BCEWithLogitsLoss`(클래스 불균형 보정 `pos_weight`)로 학습하면서 **epoch별 train/val Loss와
val AUPRC를 기록**해 그래프로 그립니다. → 학습이 실제로 수렴하는지 눈으로 확인.

> torch가 없으면 아래 셀이 안내 메시지를 출력합니다. 설치: `pip install torch` (GPU면 CUDA 빌드).

In [ ]:
try:
    import torch
    HAVE_TORCH = True
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    print("torch", torch.__version__, "| device:", DEVICE)
except ImportError:
    HAVE_TORCH = False
    print("torch 미설치 — 이 섹션은 건너뜁니다. 설치: pip install torch")

In [ ]:
# 환자별 시퀀스 + 환자 단위 라벨 로드 (Challenge 2019 PSV 원본 컬럼명 사용)
if HAVE_TORCH:
    from utils import load_patient_sequences

    CH_FEATURES = ("HR", "SBP", "DBP", "MAP", "Resp", "O2Sat", "Temp")  # PSV 실제 컬럼
    seqs, labels = load_patient_sequences(
        DATA_DIR, features=CH_FEATURES, label_col="SepsisLabel", pattern="*.psv"
    )
    if MAX_FILES:
        seqs, labels = seqs[:MAX_FILES], labels[:MAX_FILES]

    n_pos = int(np.sum(labels))
    print(f"시퀀스 {len(seqs)}개 | feature 수 F={seqs[0].shape[1]} | "
          f"양성(sepsis) {n_pos} / 음성 {len(labels) - n_pos}")
    print("예시 시퀀스 shape [T, F]:", seqs[0].shape)

In [ ]:
if HAVE_TORCH:
    from torch import nn
    from dataset import build_datasets, make_dataloader
    from model import build_model
    from sklearn.metrics import average_precision_score

    train_ds, val_ds = build_datasets(seqs, labels, val_fraction=0.2, seed=42)
    train_loader = make_dataloader(train_ds, batch_size=64, shuffle=True)
    val_loader = make_dataloader(val_ds, batch_size=128, shuffle=False)

    F = seqs[0].shape[1]
    model_rnn = build_model(input_size=F, rnn_type="lstm", hidden_size=64).to(DEVICE)

    # 클래스 불균형 보정
    n_pos_tr = float(np.sum(train_ds.labels)); n_neg_tr = len(train_ds.labels) - n_pos_tr
    pos_weight = torch.tensor([n_neg_tr / max(n_pos_tr, 1.0)], device=DEVICE)
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    opt = torch.optim.Adam(model_rnn.parameters(), lr=1e-3)

    EPOCHS = 15
    hist = {"train_loss": [], "val_loss": [], "val_auprc": []}

    for epoch in range(EPOCHS):
        model_rnn.train(); tl = 0.0; n = 0
        for b in train_loader:
            x, y, L = b["x"].to(DEVICE), b["y"].to(DEVICE), b["lengths"]
            opt.zero_grad()
            loss = loss_fn(model_rnn(x, L), y)
            loss.backward(); opt.step()
            tl += loss.item() * len(y); n += len(y)

        model_rnn.eval(); vl = 0.0; vn = 0; ys = []; ps = []
        with torch.no_grad():
            for b in val_loader:
                x, y, L = b["x"].to(DEVICE), b["y"].to(DEVICE), b["lengths"]
                logit = model_rnn(x, L)
                vl += loss_fn(logit, y).item() * len(y); vn += len(y)
                ys.append(y.cpu().numpy()); ps.append(torch.sigmoid(logit).cpu().numpy())
        yv, pv = np.concatenate(ys), np.concatenate(ps)
        auprc = float(average_precision_score(yv, pv)) if yv.sum() > 0 else float("nan")
        hist["train_loss"].append(tl / n); hist["val_loss"].append(vl / vn); hist["val_auprc"].append(auprc)
        print(f"epoch {epoch+1:2d}/{EPOCHS}  train_loss={tl/n:.4f}  val_loss={vl/vn:.4f}  val_AUPRC={auprc:.3f}")

In [ ]:
if HAVE_TORCH:
    fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
    ax[0].plot(range(1, EPOCHS + 1), hist["train_loss"], marker="o", label="train")
    ax[0].plot(range(1, EPOCHS + 1), hist["val_loss"], marker="o", label="val")
    ax[0].set_title("BCE Loss curve"); ax[0].set_xlabel("epoch"); ax[0].set_ylabel("loss"); ax[0].legend()
    ax[1].plot(range(1, EPOCHS + 1), hist["val_auprc"], marker="o", color="green")
    ax[1].set_title("Validation AUPRC"); ax[1].set_xlabel("epoch"); ax[1].set_ylabel("AUPRC")
    plt.tight_layout()
    plt.show()

## 6. 요약

- **1~2절**: 실데이터가 제대로 로드/정제되고, 결측·분포·궤적을 눈으로 확인.
- **3절**: 예측 지평이 좁으면 양성이 희박해져 AUPRC가 붕괴 — `HORIZON`을 6h 등으로 넓히면 학습 가능.
- **4절**: XGBoost vs NEWS를 PR/ROC로 비교. ROC는 비슷해도 PR(정밀도=오경보)에서 차이가 나는지 확인.
- **5절**: LSTM/GRU를 같은 데이터로 학습, **Loss 곡선이 내려가고 val AUPRC가 오르면** DL 벤치마크가
  실데이터에서 작동함을 확인.

> 다시 강조: Challenge 2019는 **패혈증** 프록시입니다. 이 숫자는 파이프라인 작동 증거이며, 경북대
> **심정지** 성능이 아닙니다. 제안서에는 합성 시연 수치로 정직하게 표기되어 있습니다.

**본실행 팁**: 전체 데이터(setA+setB)로 `MAX_FILES=None`, GPU면 `USE_GPU=True`, 지평을 6/12h로 바꿔
비교해 보세요. XGBoost 튜닝은 CLI `python src/sepsis_explore.py <dir> --horizon=6 --tune --gpu`.